In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#LIBRARIES

#system
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import sys

#math and array operations
import numpy as np
import math

#plotting
import matplotlib
# matplotlib.use("Agg") #UNCOMMENT IF PLOTTING WITHIN JUPYTER DOCUMENT
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import TwoSlopeNorm

import cartopy.crs as ccrs
import cartopy.feature as cfeature

#data classes
import xarray as xr
import h5py
import pickle 

#loading bar
from tqdm import tqdm

#dates
from datetime import datetime

In [ ]:
#Importing DirectoryManager Class
sys.path.append(os.path.join("/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/","DataAnalysis"))
from CLASSES_Directories import DirectoryManager_Class

In [ ]:
DirectoryManager = DirectoryManager_Class()

codeType = os.path.join("DataAnalysis", "MPAS_Model_Data", "VariableComparisons")
# dataType = "AreaAverages"
dataType = "AreaAverages_Interpolation" #*INTERPOLATION

outputDirectory = DirectoryManager.GetOutputDirectory(codeType, dataType)
outputPlottingDirectory = DirectoryManager.GetOutputPlottingDirectory(codeType, dataType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_ModelData import StructuredModelData_Class, DataOperator_Class

In [ ]:
#Setup

Region = "TRACER"; Case = "WET"; spinup_hours = "0"
Region = "TRACER"; Case = "DIURNAL"; spinup_hours = "-5"

Region = "PRECIP"; Case = "WET"; spinup_hours = "12"
Region = "PRECIP"; Case = "DIURNAL"; spinup_hours = "12"

Region = "Hawaii"; Case = "WET"; spinup_hours = "12"
Region = "Hawaii"; Case = "TRADES"; spinup_hours = "24" 

In [ ]:
#Load Model Directory Class
RunType = (Region,Case,"NSSL",spinup_hours)
ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

# RunType = (Region,Case,"TEMPO",spinup_hours)
# ModelData_TEMPO = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType)

In [ ]:
#Importing ModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_DataSaving import DataSaving_Class

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","MPAS_Model_Data"))
from CLASSES_PlottingModelData import RadarPlotting_Class

In [ ]:
#Importing Radar Classes
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis","Observation_Data"))
from CLASSES_RadarDataLoading import RadarData_MRMS_Class, RadarObservationMask_Class

sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_RadarDataPlotting import RadarPlotting_Class

In [ ]:
#IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys
path = os.path.join(DirectoryManager.mainCodeDirectory, 'Functions_2.0')
sys.path.append(path)


# --- Import all your function modules ---
import importlib
modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)        # import module itself
    globals().update(vars(globals()[mod]))              # import all functions into global namespace

In [ ]:
###############
#JOB ARRAY SETUP

In [ ]:
#Importing PlottingModelData Class
sys.path.append(os.path.join(DirectoryManager.mainCodeDirectory,"DataAnalysis"))
from CLASSES_JobArray import JobArray_Class

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs():
    num_jobs=20
    return num_jobs

num_jobs = GetNumJobs()
JobArray = JobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = JobArray.start_job; end_job = JobArray.end_job

def GetLoopElements(start_job,end_job):
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job].tolist()
    return loop_elements
loop_elements = GetLoopElements(start_job,end_job)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def InitiateMatrix(variableSubset, fill_nan=False):
    """
    Initializes an output matrix for a given variable subset.
    """
    if "nVertLevels" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzc)

    elif "nVertLevelsP1" in variableSubset.dims:
        shape = (ModelData.Ntime, ModelData.Nzf)

    else:  # 2D variable case
        shape = (ModelData.Ntime, 1)

    fill_value = np.nan if fill_nan else 0
    output = np.full(shape, fill_value, dtype=float)

    return output

# def GetMean(variableSubset):
#     variableMean = variableSubset.mean(dim=("latitude","longitude"), skipna=True).data
#     return variableMean
def GetMean(variableSubset):
    #(1/A) times integral of phi dA 
    #dA is [(R*cos(Lat)dLon)][RdLat] = R^2 cos(Lat)dLatdLon ==> weight is simply cos(Lat)
    weights = np.cos(np.deg2rad(variableSubset.latitude))
    variableMean = variableSubset.weighted(weights).mean(
        dim=("latitude", "longitude"),
        skipna=True
    )
    return variableMean

def MeanDBZ(variableSubset):
    # Convert from dBZ → linear Z (mm^6 m^-3)
    variableSubset_power = 10 ** (variableSubset / 10.0)

    # Take mean in linear space
    variableMean = GetMean(variableSubset_power)

    # Convert mean Z → back to dBZ
    variableMean = 10.0 * np.log10(variableMean)

    return variableMean

In [ ]:
#Loading Radar Mask
RadarDataMask = RadarObservationMask_Class.LoadMaskData(DirectoryManager, ModelData)

In [ ]:
####################################
#CALCULATION FUNCTIONS

In [ ]:
def GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName):
    """
    Retrieves a variable subset from the given model data.
    If varName contains a '+', returns the sum of the two variables.
    """
    if '+' in varName:
        var1, var2 = varName.split('+')
        var1 = var1.strip()
        var2 = var2.strip()

        subset1 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var1)
        subset2 = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                      dataSubset_diag, dataSubset_static, var2)
        variableSubset = subset1 + subset2
    else:
        variableSubset = DataOperator_Class.GetData_Variable(ModelData, dataSubset, 
                                                             dataSubset_diag, dataSubset_static, varName)

    return variableSubset

In [ ]:
def RunCalculations(ModelData, varNames, loop_elements,zTarget=None):
    outputDictionary={}
    
    for count, t in enumerate(tqdm(loop_elements, desc="Processing timesteps")):
        # if t % 10 == 0: print(f"Currently working on time {t}/{num_times}","\n")
            
        #Loading Data
        [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t)

        for varName in varNames:
            if count == 0: print(f"Running for {varName}")
                
            #Subsetting Data
            variableSubset= GetVariableSubset_Helper(ModelData, dataSubset, dataSubset_diag, dataSubset_static, varName)
            
            #Interpolating Z levels #*INTERPOLATION
            #################################
            if any(dim.startswith("nVertLevels") for dim in variableSubset.dims):
                if zTarget is None:
                    [zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c)
                    zTarget = "loaded"
                variableSubset = ModelData.InterpolateVertical(variableSubset,zGrid_f,zGrid_c,zTarget_f,zTarget_c)
            #################################            
            
            if varName in ['refl10cm','refl10cm_1km']:
                variableSubset = variableSubset.where(variableSubset > 0)

            #Applying RadarDataMask
            variableSubset = variableSubset.where(RadarDataMask == True)
            
            #Initializing Output
            if count == 0:
                output = InitiateMatrix(variableSubset, fill_nan=False)
                outputDictionary[varName] = output

            #Taking Mean
            if varName in ['refl10cm','refl10cm_1km']:
                variableMean = MeanDBZ(variableSubset)  
            else:
                variableMean = GetMean(variableSubset)
                
            outputDictionary[varName][t] = variableMean
    return outputDictionary
    
def GetData_Subset(ModelData,t):  
    data = ModelData.GetDataTimestep(t,printout=False)
    data_diag = ModelData.GetDataTimestep_diag(t,printout=False)
    
    [latCenter,lonCenter] = DataOperator_Class.LatLonBoundingBox_Center(region=ModelData.region)
    [latBounds, lonBounds] = DataOperator_Class.LatLonBoundingBox_Calculation(latCenter, lonCenter, radius_km=500)
    dataSubset, lat, lon = DataOperator_Class.LatLonBoundingBox_Subset(data,latBounds, lonBounds)
    dataSubset_diag, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(data_diag,latBounds, lonBounds)
    dataSubset_static, _, _ = DataOperator_Class.LatLonBoundingBox_Subset(ModelData.staticData,latBounds, lonBounds)

    # Lon, Lat = np.meshgrid(lon, lat) #not actually needed to plot
    return dataSubset, dataSubset_diag, dataSubset_static, lat, lon, data, data_diag

In [ ]:
def RunAreaAverages(ModelData,varNames,name, loop_elements):
    filePath = DataOperator_Class.GetOutputFilePath(ModelData, DirectoryManager, outputDirectory, fileName = f"outputDictionary_{name}_{loop_elements[0]}-{loop_elements[-1]+1}.h5")
    
    #loading back in 
    try:
        #loading output
        print("\n")
        outputDictionary = DataSaving_Class.LoadDictionaryFromH5(filePath)
        return outputDictionary
    except Exception as e:
        print(f"Error: {e}")
        
        print("Running Calculation")
        outputDictionary = RunCalculations(ModelData, varNames, loop_elements) #takes about 10 minutes
        #saving output
        print("\n")
        DataSaving_Class.SaveDictionaryToH5(outputDictionary, filePath)
        return outputDictionary

In [ ]:
####################################
#CALCULATING FUNCTIONS

In [ ]:
def GetDictionary_1(ModelData, loop_elements):
    #2D Variables (12 vars)
    #surface variables
    varNames = ["u10", "v10", "q2", "t2m", "th2m",
                "hfx", "qfx", "lh"]
    #microphysics variables
    varNames += ["refl10cm_1km","rainnc+rainc"]
    #convection variables
    varNames += ["cape", "cin"]
    outputDictionary_1 = RunAreaAverages(ModelData,varNames,"1", loop_elements)
    return outputDictionary_1

In [ ]:
def GetDictionary_2(ModelData, loop_elements):
    #3D Variables (9 vars)
    #microphysics variables
    varNames = ["qv", "qc+qi", "qs","qr", "qg", "relhum"]
    #convection variables
    varNames += ["w", "theta"]
    
    outputDictionary_2 = RunAreaAverages(ModelData,varNames,"2", loop_elements)
    return outputDictionary_2

In [ ]:
def RunJob(loop_elements):
    
    #getting NSSL dictionaries
    RunType = (Region,Case,"NSSL",spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, printSummary=False)
    
    outputDictionary_2D_NSSL = GetDictionary_1(ModelData, loop_elements)
    outputDictionary_3D_NSSL = GetDictionary_2(ModelData, loop_elements)
    
    #getting TEMPO dictionaries
    RunType = (Region,Case,"TEMPO",spinup_hours)
    ModelData = StructuredModelData_Class(DirectoryManager.mainDirectory, DirectoryManager.scratchDirectory, RunType, printSummary=False)
    
    outputDictionary_2D_TEMPO = GetDictionary_1(ModelData, loop_elements)
    outputDictionary_3D_TEMPO = GetDictionary_2(ModelData, loop_elements)

    return outputDictionary_2D_NSSL,outputDictionary_3D_NSSL,outputDictionary_2D_TEMPO,outputDictionary_3D_TEMPO

In [ ]:
####################################
#CALCULATING
running = True #keep true when job_array is running
running = False

In [ ]:
if running:
    [outputDictionary_2D_NSSL,outputDictionary_3D_NSSL,outputDictionary_2D_TEMPO,outputDictionary_3D_TEMPO] = RunJob(loop_elements)

In [ ]:
####################################
#RECOMBINING
recombining = False #keep false when job_array is running
recombining = True

In [ ]:
def AddDictionaries(dictA, dictB):
    """
    Modifies dictA by adding dictB values into it
    """
    for key in dictA:
        dictA[key] += dictB[key]
        
def Recombine():
    for job_id in tqdm(range(1, num_jobs + 1)):
    
        start_job, end_job = JobArray._get_job_range(job_id)
        loop_elements = GetLoopElements(start_job, end_job)
    
        dict2D_NSSL, dict3D_NSSL, dict2D_TEMPO, dict3D_TEMPO = RunJob(loop_elements)
    
        if job_id == 1:
            dict2D_NSSL_all, dict3D_NSSL_all = dict2D_NSSL, dict3D_NSSL
            dict2D_TEMPO_all, dict3D_TEMPO_all = dict2D_TEMPO, dict3D_TEMPO
        else:
            AddDictionaries(dict2D_NSSL_all,  dict2D_NSSL)
            AddDictionaries(dict3D_NSSL_all,  dict3D_NSSL)
            AddDictionaries(dict2D_TEMPO_all, dict2D_TEMPO)
            AddDictionaries(dict3D_TEMPO_all, dict3D_TEMPO)
    return dict2D_NSSL_all,dict3D_NSSL_all, dict2D_TEMPO_all,dict3D_TEMPO_all

#extrapolating th2m for time 0 
def FixTheta2m(outputDictionary_2D_NSSL,outputDictionary_2D_TEMPO):
    for a in [outputDictionary_2D_NSSL,outputDictionary_2D_TEMPO]:
        a['th2m'][0] = 2 * a['th2m'][1] - a['th2m'][2]

In [ ]:
if recombining:
    [outputDictionary_2D_NSSL,outputDictionary_3D_NSSL,outputDictionary_2D_TEMPO,outputDictionary_3D_TEMPO] = Recombine()
    FixTheta2m(outputDictionary_2D_NSSL,outputDictionary_2D_TEMPO)

In [ ]:
####################################
#PLOTTING FUNCTIONS

fontSettings = {
    "tickFont": 16-4,
    "labelFont": 18-4,
    "legendFont": 14-2,
    "titleFont": 22,
}

plotting = False #keep false when job array is running
plotting = True

In [ ]:
def GetVerticalMean(a, 
                    axis,
                    dataType=""):
    if dataType == "reflectivity":
        pass
        # # Convert from dBZ → linear Z (mm^6 m^-3)
        # Z_linear = 10 ** (a / 10.0)

        # # Take mean in linear space
        # Z_mean = np.nanmean(Z_linear, axis=axis)

        # # Convert mean Z → back to dBZ
        # dBZ_mean = 10.0 * np.log10(Z_mean)
        # return dBZ_mean
    else:
        return np.nanmean(a, axis=axis)

def verticalCollapse(a, 
                     axis=1,km_level=6, 
                     collapseType="z_mean"): #currently slicing at 3 km, no average
    if collapseType == "z_mean":
        return GetVerticalMean(a,axis)
    if "z" in collapseType:
        #Getting Z Threshold Indexes
        # z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
        # zlevels = np.loadtxt(z_levels_filePath)/1e3
        # zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
        # zlevels = zTarget_f/1e3 #*INTERPOLATION
        zlevels_center = zTarget_c/1e3 #*INTERPOLATION
        z_idx = np.argmin(np.abs(zlevels_center - km_level))
        if collapseType == "z_slice":
            return a[:,z_idx]
        elif collapseType == "z_slice_upto":
            return GetVerticalMean(a[:, :z_idx+1],axis)

In [ ]:
def GetVerticalCoord(dataSubset):
    pressure_profile = dataSubset['pressure'].mean(dim=("latitude","longitude")).data
    dp = pressure_profile[-1] - pressure_profile[-2]
    p_topface = pressure_profile[-1] + dp  # extrapolate linearly
    pressure_profile_face = np.append(pressure_profile, p_topface)
    return (pressure_profile/100,pressure_profile_face/100)

if plotting:
    [dataSubset,dataSubset_diag,dataSubset_static, lat,lon,zGrid_f,zGrid_c, _, _] = DataOperator_Class.GetData_Subset(ModelData, t=0)
    pressure_profiles = GetVerticalCoord(dataSubset)
    time_strings = ModelData.timeStrings
    time = [datetime.strptime(t, "%Y-%m-%d_%H.%M.%S") for t in time_strings]

In [ ]:
#Helper Functions

def nansubtract(a, b):
    """
    Element-wise subtraction (a - b) that preserves NaNs.

    If shapes differ, raises a ValueError.
    """
    if a.shape != b.shape:
        raise ValueError(f"Shape mismatch: a{a.shape} != b{b.shape}")

    return np.where(np.isnan(a) | np.isnan(b), np.nan, a - b)

# Example: align datetime x-limits to min/max of your data
def SetXLimitsDatetime(ax, time_array):
    """
    Ensures datetime x-axis starts and ends exactly at the first and last time values.
    Works for both datetime.datetime and np.datetime64 arrays.
    """
    import numpy as np
    from matplotlib.dates import date2num

    # Convert to Matplotlib’s internal float format if needed
    times = np.asarray(time_array)
    if np.issubdtype(times.dtype, np.datetime64):
        times = date2num(times)
    elif isinstance(times[0], (object,)):
        try:
            times = date2num(times)
        except Exception:
            pass

    ax.set_xlim(times.min(), times.max())

def AlignAxesRight(ax_list):
    """
    Aligns the right edges of all axes in ax_list (e.g., contour + line plots),
    so that colorbars don't make some axes narrower.

    It uses the first axis that contains a contour or image
    (typically a contourf plot) as the reference width.
    """

    # Try to find a contour axis (has .collections or .images)
    ref_ax = None
    for ax in ax_list:
        if getattr(ax, "collections", []) or getattr(ax, "images", []):
            ref_ax = ax
            break

    # If no contour axis found, just use the first axis
    if ref_ax is None:
        ref_ax = ax_list[0]

    ref_pos = ref_ax.get_position()

    # Apply its width to all other axes
    for ax in ax_list:
        pos = ax.get_position()
        new_pos = [pos.x0, pos.y0, ref_pos.width, pos.height]
        ax.set_position(new_pos)

    print(f"Aligned {len(ax_list)} axes using reference width from contour axis at {ref_pos.width:.3f}")
# #EXAMPLE USAGE
# fig, axs = plt.subplots(2, 1, figsize=(8, 6))

# # contourf on top, line on bottom
# time = np.arange(24)
# pressure = np.linspace(1000, 100, 25)
# data = np.sin(time / 3)[None, :] * np.exp(-pressure[:, None] / 1000)

# plot = axs[0].contourf(time, pressure, data, cmap="RdBu_r")
# plt.colorbar(plot, ax=axs[0], orientation="vertical", pad=0.02)
# axs[1].plot(time, np.sin(time / 3), color="k")

# # Align both
# AlignAxesRight(axs)

# plt.show()

from matplotlib.ticker import MultipleLocator
def add_minor_white_grid(ax, alpha=0.5, lw=1.0, thickness=1.4, color='lightgray',
                         applyX=True,applyY=False):
    """
    Add white semi-transparent grid lines halfway between major ticks
    on both x and y axes (for contour plots).
    """
    from matplotlib.ticker import MultipleLocator

    # --- Minor locators at half the major spacing ---
    if applyX:
        major_x = ax.xaxis.get_major_locator()
        step_x = major_x()[1] - major_x()[0]
        ax.xaxis.set_minor_locator(MultipleLocator(step_x / 2))
    if applyY:
        major_y = ax.yaxis.get_major_locator()
        step_y = major_y()[1] - major_y()[0]
        ax.yaxis.set_minor_locator(MultipleLocator(step_y / 2))

    # --- Grid styling ---
    ax.grid(True, which="major", color=color, alpha=alpha, lw=lw * thickness)
    ax.grid(True, which="minor", color=color, alpha=alpha, lw=lw)

def AdjustLayout(fig,
                 left=0.07, right=0.97, bottom=0.07,
                 wspace=0.35, hspace=0.6,
                 title_space_inches=0.9, 
                 title_y_inches_from_top=0.25):
    """
    Applies a robust manual Matplotlib layout
    to a figure, reserving absolute space for a suptitle.
    """
    
    # Get figure height in inches
    fig_height_inches = fig.get_figheight()
    
    # Calculate the 'top' margin (where plots end) in relative figure coords
    # This leaves 'title_space_inches' at the top.
    top_margin = 1.0 - (title_space_inches / fig_height_inches)
    
    # Calculate the 'y' position for the suptitle
    title_y_relative = 1.0 - (title_y_inches_from_top / fig_height_inches)
    
    # Apply the manual layout
    plt.subplots_adjust(left=left, right=right, bottom=bottom, 
                        top=top_margin, wspace=wspace, hspace=hspace)

    # Return the calculated 'y' coordinate for the suptitle
    return title_y_relative

In [ ]:
##GetColormap

from matplotlib import cm
# import seaborn as sns #sns.color_palette("mako", as_cmap=True)

cmapDictionary = {
    # signed / diverging
    "w": cm.RdBu_r,
    "divergence": cm.RdBu_r,

    # moisture / hydrometeors
    "qv": cm.turbo,
    "qc+qi": cm.turbo,
    "qr": cm.turbo,
    "qg": cm.turbo,
    "qs": cm.turbo,

    # thermodynamic
    "theta": cm.turbo,

    # bounded scalar
    "relhum": cm.turbo,

    # reflectivity (handled elsewhere)
    "refl10cm": None,
    "refl10cm_1km": None,
}

def GetColormap(varName):
    return cmapDictionary.get(varName, cm.viridis)


In [ ]:
# PlotSingle

zGrid_f, zGrid_c = ModelData.GetZGrids() #*INTERPOLATION
[zTarget_f, zTarget_c] = ModelData.GetZTarget(zGrid_f, zGrid_c) #*INTERPOLATION

def PlotSingle(axis, outputDictionarys, varName, time, pressure_profiles, labels,
               plottype="TZ", clim=None,num_levels=17,cbar=None):

    """
    Plot one variable on a given Matplotlib axis.
    Supports either:
      - A single dictionary (for single-model plots)
      - Two dictionaries (for model comparisons or line overlays)
    clim: tuple (vmin, vmax) for consistent color scaling (ignored for reflectivity)
    """

    # ------------------------------------------------------
    #  Helper: Line Plot
    # ------------------------------------------------------
    def lineplot(time, output, varName, units, color, label):
        axis.plot(time, output.squeeze(), color=color, label=label)
        axis.set_ylabel(f"{varName} " + fr"$({units})$")
        axis.set_xlabel("Time")
        axis.grid(True)
        SetXLimitsDatetime(axis, time)
    def lineplot2(output,zlevels_plot, varName, units, color, label):
        axis.plot(output.squeeze(),zlevels_plot, color=color, label=label)
        axis.set_xlabel(f"{varName} " + fr"$({units})$")
        axis.set_ylabel("z (km)")
        axis.grid(True)
    
    # ------------------------------------------------------
    #  Helper: Consistent Colorbar Formatting
    # ------------------------------------------------------
    def add_colorbar(fig, mappable, ax, varName,label, ticks=None, orientation="vertical"):
        """Add a consistently styled, larger colorbar."""
        cbar = fig.colorbar(
            mappable, ax=ax, orientation=orientation,
            fraction=0.12, pad=0.020, aspect=20, shrink=1.15
        )
        cbar.set_label(label, fontsize=fontSettings["labelFont"])
        cbar.ax.tick_params(labelsize=8, width=1.1, length=4, pad=2)
        if ticks is not None:
            cbar.set_ticks(ticks)
        # Prevent overcrowding
        if len(cbar.get_ticks()) > 10:
            # from matplotlib.ticker import MaxNLocator #depreciated
            # cbar.ax.yaxis.set_major_locator(MaxNLocator(8)) #depreciated
            vmin, vmax = mappable.get_clim()
            auto_ticks = np.linspace(vmin, vmax, 9)
            cbar.set_ticks(auto_ticks)

        cbar.ax.tick_params(axis="both", labelsize=fontSettings["tickFont"])
        # cbar.ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))
        return cbar

    # ------------------------------------------------------
    #  Units and scaling
    # ------------------------------------------------------
    [axisTitle, multiplier, units] = GetUnitsAndScaling(varName)
    
    # ------------------------------------------------------
    #  Select pressure profile
    # ------------------------------------------------------
    # sample_dict = outputDictionarys[0]
    # output_sample = sample_dict[varName]
    # pressure_profile = (
    #     pressure_profiles[0]
    #     if output_sample.shape[1] == pressure_profiles[0].shape[0]
    #     else pressure_profiles[1]
    # )
    z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
    zlevels = np.loadtxt(z_levels_filePath)/1e3
    if varName not in ['w']:
        # zlevels_plot = 0.5 * (zlevels[:-1] + zlevels[1:])
        zlevels_plot = zTarget_c/1e3 #*INTERPOLATION
    else:
        # zlevels_plot = zlevels.copy()
        zlevels_plot = zTarget_f/1e3 #*INTERPOLATION

    # ------------------------------------------------------
    #  Case 1: Single-model plotting
    # ------------------------------------------------------
    if len(outputDictionarys) == 1:
        output = multiplier * outputDictionarys[0][varName]

        # Choose color setup
        cmap = GetColormap(varName)

        # --- Line plot ---
        if output.ndim == 1 or output.shape[1] == 1:
            color = "k"
            label = labels[0] if labels else None
            lineplot(time, output, axisTitle, units, color, label)

        # --- Contour plot ---
        else:
            if plottype == "TZ" and varName not in ["refl10cm", "refl10cm_1km"]:
                # Apply shared clim via levels
                if clim is not None:
                    c0 = multiplier * clim[0]
                    c1 = multiplier * clim[1]
                    cmin, cmax = sorted([c0, c1])
                    levels = np.linspace(cmin, cmax, num_levels)
                    # levels = multiplier*np.linspace(clim[0], clim[1], num_levels)
                else:
                    levels = num_levels

                # Symmetric norm for diverging fields
                if varName in ["w", "divergence"]:
                    norm = TwoSlopeNorm(vcenter=0.0, vmin=clim[0] if clim else np.nanmin(output),
                                        vmax=clim[1] if clim else np.nanmax(output))
                else:
                    norm = None

                plot = axis.contourf(time, zlevels_plot, output.T, cmap=cmap,
                                     levels=levels, norm=norm, extend="both")
                cbar = add_colorbar(axis.figure, plot, axis, varName,
                                    label=f"{axisTitle} " + fr"$({units})$")
                
                add_minor_white_grid(axis)
                axis.set_ylabel("Altitude (km)", fontsize=fontSettings['labelFont'])
                axis.set_xlabel("Time", fontsize=fontSettings['labelFont'])
                axis.set_ylim(0,maxZLevel)
                axis.set_yticks(np.arange(0, maxZLevel + 0.1, 2))
                # axis.invert_yaxis()

            elif plottype == "TZ" and varName in ["refl10cm", "refl10cm_1km"]:
                cmap, norm, levels, ticks = RadarPlotting_Class.GetReflectivityColormap()
                plot = axis.contourf(time, zlevels_plot, output.T,
                                     levels=levels, cmap=cmap, norm=norm, extend='both')
                cbar = add_colorbar(axis.figure, plot, axis,
                                    varName,label="Reflectivity (dBZ)", ticks=ticks)
                add_minor_white_grid(axis)
                RadarPlotting_Class.FormatReflectivityColorbar(
                    cbar, ticks, orientation='vertical', show_labels=False
                )
                axis.set_ylabel("Altitude (km)", fontsize=fontSettings['labelFont'])
                axis.set_xlabel("Time", fontsize=fontSettings['labelFont'])
                axis.set_ylim(0,maxZLevel)
                # axis.invert_yaxis()

            elif plottype == "T":
                mean_output = verticalCollapse(output, axis=1)#np.nanmean(output, axis=1)
                color = "k"
                label = labels[0] if labels else None
                lineplot(time, mean_output, axisTitle, units, color, label)
            elif plottype == "Z":
                mean_output = verticalCollapse(output, axis=0)#np.nanmean(output, axis=0)
                color = "k"
                label = labels[0] if labels else None
                lineplot2(mean_output,zlevels_plot, varName, units, color, label)
                top = 1 if varName == "qv" else 3 if varName == "theta" else maxZLevel
                axis.set_ylim(0, top)
                # if varName in ["theta","qv"]:
                #     SetXLimWithBuffer(axis, mean_output, zlevels_plot) 

    # ------------------------------------------------------
    #  Case 2: Two-model plotting
    # ------------------------------------------------------
    else:
        output1 = multiplier * outputDictionarys[0][varName]
        output2 = multiplier * outputDictionarys[1][varName]
        label1, label2 = labels

        is_line = (
            output1.ndim == 1 and output2.ndim == 1
            or output1.shape[1] == 1 and output2.shape[1] == 1
        )

        if is_line:
            with np.errstate(invalid="ignore"):
                # mean1 = np.nanmean(output1, axis=1) if output1.ndim > 1 else output1
                mean1 = verticalCollapse(output1, axis=1) if output1.ndim > 1 else output1


                
                # mean2 = np.nanmean(output2, axis=1) if output2.ndim > 1 else output2
                mean2 = verticalCollapse(output2, axis=1) if output2.ndim > 1 else output2
            lineplot(time, mean1, axisTitle, units, "blue", label1)
            lineplot(time, mean2, axisTitle, units, "green", label2)
            axis.legend(loc="upper left")

        else:
            if plottype == "TZ":
                diff = nansubtract(output1, output2)
                cmap = plt.get_cmap("RdBu_r").copy()
                # cmap.set_bad("black")
                # axis.set_facecolor('black')
                # vlim = np.nanmax(np.abs(diff))
                vlim = np.nanpercentile(np.abs(diff), 99) #using 99% percentile instead of max
                levels=np.linspace(-vlim, vlim, num_levels)
                norm = TwoSlopeNorm(vcenter=0.0, vmin=-vlim, vmax=vlim)
                plot = axis.contourf(time, zlevels_plot, diff.T, cmap=cmap,
                                     levels=levels,
                                     norm=norm, extend="both")
                cbar = add_colorbar(axis.figure, plot, axis, 
                                    varName,label=f"Δ{axisTitle} " + fr"$({units})$")
                axis.set_ylabel("Altitude (km)", fontsize=fontSettings['labelFont'])
                axis.set_xlabel("Time", fontsize=fontSettings['labelFont'])
                axis.set_ylim(0,maxZLevel)
                axis.set_yticks(np.arange(0, maxZLevel + 0.1, 2))
                # axis.invert_yaxis()

            elif plottype == "T":
                with np.errstate(invalid="ignore"):
                    mean1 = verticalCollapse(output1, axis=1)#np.nanmean(output1, axis=1)
                    mean2 = verticalCollapse(output2, axis=1)#np.nanmean(output2, axis=1)
                lineplot(time, mean1, axisTitle, units, "blue", label1)
                lineplot(time, mean2, axisTitle, units, "green", label2)
                axis.legend(loc="upper left")

            elif plottype == "Z":
                with np.errstate(invalid="ignore"):
                    mean1 = verticalCollapse(output1, axis=0)#np.nanmean(output1, axis=0)
                    mean2 = verticalCollapse(output2, axis=0)#np.nanmean(output2, axis=0)
                lineplot2(mean1,zlevels_plot, varName, units, "blue", label)
                lineplot2(mean2,zlevels_plot, varName, units, "green", label)
                axis.legend(loc="upper right")
                top = 1 if varName == "qv" else 3 if varName == "theta" else maxZLevel
                axis.set_ylim(0, top)
                # if varName in ["theta","qv"]:
                #     combinedMean = np.concatenate([mean1, mean2])
                #     SetXLimWithBuffer(axis, combinedMean, zlevels_plot)
                
                
    if varName in ["qr","qs","qg","qc+qi","w"]:
        if plottype=="T":
            apply_scientific_notation([axis],dim='y')
        elif plottype=="Z":
            apply_scientific_notation([axis],dim='x')
    apply_scientific_notation_colorbar([cbar])
    
    # ------------------------------------------------------
    #  Title and finish
    # ------------------------------------------------------
    axis.set_title(axisTitle, fontsize=11)

    axis.tick_params(axis="both", labelsize=fontSettings["tickFont"])

def SetXLimWithBuffer(axis, xdata, ydata=None, buffer=0.05):
    """
    Set x-limits with a fractional buffer based on data range.
    """

    xdata = np.asarray(xdata)

    # ---- optionally filter by current y-limits ----
    if ydata is not None:
        ymin, ymax = axis.get_ylim()
        ydata = np.asarray(ydata)
        mask = (ydata >= ymin) & (ydata <= ymax)
        xdata = xdata[mask]

    # ---- remove NaNs ----
    xdata = xdata[np.isfinite(xdata)]

    if xdata.size == 0:
        return

    xmin = xdata.min()
    xmax = xdata.max()

    dx = xmax - xmin
    pad = buffer * dx if dx > 0 else buffer * max(abs(xmax), 1e-12)

    axis.set_xlim(xmin - pad, xmax + pad)

def GetUnitsAndScaling(varName):
    """
    Returns (multiplier, units, axisTitle) for a given variable name.
    """
    units = ModelData.GetUnits_Specific(varName).replace(" ", r"\ ")
    axisTitle = varName

    mixingRatioTitles = {"qv":    r"$q_v$",
                         "qc":    r"$q_c$", "qi":    r"$q_i$", "qc+qi": r"$q_c + q_i$",
                         "qr":    r"$q_r$",
                         "qg":    r"$q_g$",
                         "qs":    r"$q_s$"}

    if varName in ["qv", "qc", "qi", "qc+qi", "qr", "qg", "qs"]:
        multiplier = 1e3
        units = units.replace('kg', 'g', 1)
        axisTitle = mixingRatioTitles[varName]
    
    elif varName == "divergence":
        multiplier = -1
        axisTitle = "convergence"
    elif varName == "relhum":
        multiplier = 1
        axisTitle = "RH"
        units = units.replace("percent", r"\%")
    elif varName == "theta":
        multiplier = 1
        axisTitle = r"$\theta$"
    else:
        multiplier = 1

    return axisTitle, multiplier, units

In [ ]:
def MakeCombinedPlot(combinedDict, key, plottype):
    """
    Creates a combined plot from a model-comparison dictionary:
    combinedDict = {"NSSL": {...}, "TEMPO": {...}, ...}

    For two models and TZ plots:
      Each row = variable
      Columns = [Model1, Model2, Difference]
    For T plots:
      Both models are overlaid on the same axes with distinct colors.
    """

    # --- Extract model and variable structure ---
    combinedDict2 = combinedDict[key]
    modelLabels = list(combinedDict2.keys())  # e.g. ["NSSL", "TEMPO"]
    first_model = modelLabels[0]
    varNames = list(combinedDict2[first_model].keys())
    # --- Force "divergence" (convergence) to be last ---
    if "divergence" in varNames:
        varNames = [v for v in varNames if v != "divergence"] + ["divergence"]
    n_vars = len(varNames)

    # --- Layout logic ---
    if len(modelLabels) == 2 and plottype == "TZ":
        n_cols = 3  # Model1, Model2, Difference
        n_rows = n_vars
        layout_mode = "comparison"
    else:
        n_cols = 3
        n_rows = int(np.ceil(n_vars / n_cols))
        layout_mode = "overlay"

    fig = plt.figure(figsize=(5.5 * n_cols, 3.5 * n_rows))
    gs = gridspec.GridSpec(n_rows, n_cols, figure=fig, wspace=0.2, hspace=0.6) #wspace=0.3, hspace=0.6)

    # --- Loop through variables ---
    
    for i, varName in enumerate(varNames):
        [axisTitle,_,_,] = GetUnitsAndScaling(varName)
        # ============================================================
        # TZ layout: 3 columns per variable (Model1, Model2, Δ)
        # ============================================================
        if layout_mode == "comparison":
            row = i

            # --- Compute shared clim for both models ---
            if varName not in ["refl10cm", "refl10cm_1km"]:
                out1 = combinedDict2[modelLabels[0]][varName]
                out2 = combinedDict2[modelLabels[1]][varName]
                vmin = np.nanmin([np.nanmin(out1), np.nanmin(out2)])
                vmax = np.nanmax([np.nanmax(out1), np.nanmax(out2)])
                if varName in ["w","divergence"]: #w
                    vlim = np.nanmax(np.abs([vmin, vmax]))
                    clim = (-vlim, vlim)
                else:
                    clim = (vmin, vmax)
            else:
                clim = None

            # --- Column 1: Model 1 ---
            ax1 = fig.add_subplot(gs[row, 0])
            PlotSingle(ax1, [combinedDict2[modelLabels[0]]], varName, time, pressure_profiles,
                       labels=[modelLabels[0]], plottype=plottype, clim=clim)
            ax1.set_title(f"{modelLabels[0]} {axisTitle}", fontsize=fontSettings['labelFont'])

            # --- Column 2: Model 2 ---
            ax2 = fig.add_subplot(gs[row, 1])
            PlotSingle(ax2, [combinedDict2[modelLabels[1]]], varName, time, pressure_profiles,
                       labels=[modelLabels[1]], plottype=plottype, clim=clim)
            ax2.set_title(f"{modelLabels[1]} {axisTitle}", fontsize=fontSettings['labelFont'])
            ax2.set_ylabel("")

            # --- Column 3: Difference ---
            ax3 = fig.add_subplot(gs[row, 2])
            PlotSingle(ax3, [combinedDict2[modelLabels[0]], combinedDict2[modelLabels[1]]],
                       varName, time, pressure_profiles,
                       labels=modelLabels, plottype=plottype)
            ax3.set_title(f"Δ({modelLabels[0]} - {modelLabels[1]}) {axisTitle}", fontsize=fontSettings['labelFont'])
            ax3.set_ylabel("")

        # ============================================================
        # T layout: overlay both models on same axes (line plots)
        # ============================================================
        # elif layout_mode == "overlay":
        #     row, col = divmod(i, n_cols)
        #     ax = fig.add_subplot(gs[row, col])

        #     # define consistent colors for models
        #     model_colors = {"NSSL": "blue", "TEMPO": "green"}

        #     # plot both models on same axis
        #     for label in modelLabels:
        #         dataDictionary = combinedDict2[label]

        #         # track existing lines to only recolor new ones
        #         existing_lines = len(ax.get_lines())
        #         # PlotSingle(ax, [dataDictionary], varName, time, pressure_profiles,
        #         #            labels=[label], plottype="T")
        #         PlotSingle(ax, [dataDictionary], varName, time, pressure_profiles,
        #                    labels=[label], plottype=plottype)
        #         new_lines = ax.get_lines()[existing_lines:]

        #         for line in new_lines:
        #             line.set_color(model_colors.get(label, "k"))
        #             line.set_label(label)

        #     ax.legend(loc="upper left", fontsize=9)
        #     ax.set_title(axisTitle, fontsize=11)
    
        elif layout_mode == "overlay":
            row, col = divmod(i, n_cols)
            ax = fig.add_subplot(gs[row, col])
        
            model_colors = {"NSSL": "blue", "TEMPO": "green"}
        
            # ---- NEW: containers for x/y data ----
            xdata_for_xlim = []
            ydata_for_xlim = []
        
            # plot both models on same axis
            for label in modelLabels:
                dataDictionary = combinedDict2[label]
        
                existing_lines = len(ax.get_lines())
                PlotSingle(
                    ax, [dataDictionary], varName, time, pressure_profiles,
                    labels=[label], plottype=plottype
                )
                new_lines = ax.get_lines()[existing_lines:]
        
                for line in new_lines:
                    line.set_color(model_colors.get(label, "k"))
                    line.set_label(label)
        
                    # ---- NEW: grab plotted data ----
                    xdata_for_xlim.append(line.get_xdata())
                    ydata_for_xlim.append(line.get_ydata())
        
            ax.legend(loc="upper left", fontsize=fontSettings['legendFont'])
            ax.set_title(axisTitle, fontsize=fontSettings['labelFont'])
        
            # ---- NEW: set x-limits once, after plotting ----
            if xdata_for_xlim and plottype=="Z":
                combinedX = np.concatenate(xdata_for_xlim)
                combinedY = np.concatenate(ydata_for_xlim)
                SetXLimWithBuffer(ax, combinedX, combinedY)
    

    # ============================================================
    # Format axes and layout
    # ============================================================
    for ax in fig.get_axes():
        ax.tick_params(labelbottom=True)
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

    title_y_relative = AdjustLayout(fig)

    # global title
    label_text = " vs ".join(modelLabels)
    # plt.suptitle(f"{ModelData.region}_{ModelData.case} {label_text}",
    #              fontsize=16, fontweight="bold", y=title_y_relative)

    # ============================================================
    # Saving Figure
    # ============================================================
    # save figure
    label_text = label_text.replace(" ", "")
    return fig, combinedDict2, key, label_text


In [ ]:
#Removing Levels Above maxZLevel km from Timeseries Averages

maxZLevel = 16
def SubsetAltitude(Dictionary,
                   maxZLevel=16):

    for varName, dataArray in Dictionary.items(): 
        
        #Getting Z Threshold Indexes
        z_levels_filePath = "/glade/derecho/scratch/aroseman/Projects/Regional-MPAS-Project/MPAS_Atmosphere_8.3.1/TRACER/WET/MPAS-Model_8.3.1_56nz/zeta_30km_57levels.txt"
        # zlevels = np.loadtxt(z_levels_filePath)/1e3
        # zlevels_center = 0.5 * (zlevels[:-1] + zlevels[1:])
        zlevels = zTarget_f/1e3 #*INTERPOLATION
        zlevels_center = zTarget_c/1e3 #*INTERPOLATION

        zc_level = np.where(zlevels_center>maxZLevel)[0][0]
        zf_level = np.where(zlevels>maxZLevel)[0][0]
        z_level = zf_level if varName == "w" else zc_level
        
        #Applying Nan to Altitudes Greater than maxZLevel km
        
        Dictionary[varName][:,z_level+1:] = np.nan

    return Dictionary

if plotting:
    outputDictionary_3D_NSSL = SubsetAltitude(outputDictionary_3D_NSSL, maxZLevel)
    outputDictionary_3D_TEMPO = SubsetAltitude(outputDictionary_3D_TEMPO, maxZLevel)

In [ ]:
def SaveFigure(fig, combinedDict, key,label_text,
               dpi=300):
    """
    Saves a figure to the appropriate directory based on the models in combinedDict.
    """
    # --- Define output subdirectory and file path ---
    outputSubDirectory = f"{ModelData.region}_{ModelData.case}_{ModelData.spinup_hours}hrs"
    os.makedirs(os.path.join(outputPlottingDirectory, outputSubDirectory), exist_ok=True)

    outputFile = os.path.join(
        outputPlottingDirectory,
        outputSubDirectory,
        f"AreaAverages_{key}.png"
    )

    # --- Save figure ---
    fig.savefig(outputFile, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved to {outputFile}")

In [ ]:
####################################
#PLOTTING

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 300

In [ ]:
if plotting:
    #REMOVING VARIABLES NOT INTERESTED IN
    # varNames = ["relhum"] #"divergence"
    # for varName in varNames:
    #     outputDictionary_3D_NSSL.pop(varName)
    #     outputDictionary_3D_TEMPO.pop(varName)
    varNames = ["rainnc+rainc","refl10cm_1km"]
    for varName in varNames:
        outputDictionary_2D_NSSL.pop(varName)
        outputDictionary_2D_TEMPO.pop(varName)

In [ ]:
if plotting:
    # setting up dictionaries for plotting
    labels = ("NSSL", "TEMPO")
    
    # variable groups
    print('varGroups')
    #list out varNames for dimension and number plot containing certain variables to plot
    varGroups = {
        # "T_1": outputDictionary_2D_NSSL.keys(),
        # "T_2": outputDictionary_3D_NSSL.keys(),
        "T_1": list(outputDictionary_2D_NSSL.keys()) + list(outputDictionary_3D_NSSL.keys()),
        "TZ_1": ["w","qc+qi","qg","qs"],
        "TZ_2": ["qr","qv","relhum","theta"],
        "TZ_3": ["w","qc+qi","qg","qs","qr","qv","relhum","theta"]
    }
    # model source dictionaries
    print('modelDicts')
    modelDicts = {
        "NSSL": [outputDictionary_2D_NSSL, outputDictionary_3D_NSSL],
        "TEMPO": [outputDictionary_2D_TEMPO, outputDictionary_3D_TEMPO],
    }
    
    # --- function ---
    def GetCombinedDictionary(varNames, dicts):
        """Get varName: value from the first dict in `dicts` that contains it."""
        #Format of combinedDictionary
        # Combined[key] = { "NSSL": {...}, "TEMPO": {...} }
        return {v: next((d[v] for d in dicts if v in d), None) for v in varNames}
    
    # --- build labeled combined structure ---
    print('combinedDictionary')
    combinedDictionary = {
        group: {
            label: GetCombinedDictionary(varNames, modelDicts[label])
            for label in labels
        }
        for group, varNames in varGroups.items()
    }

    #zeroing out first z-level, qc+qi, and qr.
    #for the casess PRECIP/DIURNAL and Hawaii/WET
    #due to anomalous near surface cloud and rain,
    #related to *issues with interpolation* of data at those regions
    #with clouds and rain along low-level slopes
    print("zeroing out bottom zlevel for qcqi and qr")
    for varName in ['qc+qi']:
        combinedDictionary['TZ_1']['NSSL'][varName][:,0]=0
        combinedDictionary['TZ_1']['TEMPO'][varName][:,0]=0
    for varName in ['qr']:
        combinedDictionary['TZ_2']['NSSL'][varName][:,0]=0
        combinedDictionary['TZ_2']['TEMPO'][varName][:,0]=0

In [ ]:
if plotting:
    #2D Variable Plots
    [fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, key = "T_1", plottype="T")
    SaveFigure(fig, combinedDict2, key, label_text)

In [ ]:
if plotting:
    #3D Variable Plots
    [fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, key = "TZ_1", plottype="TZ")
    SaveFigure(fig, combinedDict2, key, label_text,dpi=300)

In [ ]:
if plotting:
    #3D Variable Plots
    [fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, key = "TZ_2", plottype="TZ")
    SaveFigure(fig, combinedDict2, key, label_text,dpi=300)

In [ ]:
if plotting:
    #3D Variable Plots
    [fig, combinedDict2, key, label_text] = MakeCombinedPlot(combinedDictionary, key = "TZ_3", plottype="TZ")
    SaveFigure(fig, combinedDict2, key, label_text,dpi=300)